In [23]:
%pip install numpy matplotlib ipython ipykernel scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt


# Завдання 1

## Опис

Нехай $ \omega_1 $ та $ \omega_2 $ - це незалежні рівномірно розподілені на $ [0, 1] $ випадкові величини (в.в.), які отримаємо вбудованим генератором випадкових величин.
Пара незалежних в.в. $ ( \xi_1, \xi_2 ) $, які мають стандартний нормальний розподіл (тобто $ N(0, 1) $), генерується за допомогою перетворення:
$$ \xi_1 = \sqrt{-2 \ln{\omega_1}} \sin{(2 \pi \omega_2)}, \xi_2 = \sqrt{-2 \ln{\omega_1}} \cos{(2 \pi \omega_2)} $$
(в.в. $ N(0, 1) $  можна генерувати і за допомогою вбудованого генератора). Позначимо $ a = M(\xi_i) = 0, \sigma^2 = D(\xi_i) = 1 $.

Нехай спостерігається вибірка $ \bar{X} = \left( X_1, K, X_n \right)$, де $ X_i: N(0, 1) $.

Побудувати довірчий інтервал для:
- математичного сподівання $ a $  у припущенні, що спостерігаються в.в. $ \{ X_i \} $, які мають нормальний розподіл, але дисперсія $ \sigma^2 $ невідома;
- математичного сподівання $ a $ у припущенні, що спостерігаються в.в. $ \{ X_i \} $, розподіл яких невідомий.
- дисперсії $ \sigma^2 $ у припущенні, що спостерігаються в.в. $ \{ X_i \} $, які мають нормальний розподіл.

Всі довірчі інтервали будуються із достовірністю $ 1 - \gamma = 0.99 $ для $ n = 10^2 $, $ n = 10^4 $ та $ n = 10^6 $.
В усіх цих випадках дослідити, чи потрапляють математичне сподівання та дисперсія у побудовані довірчі інтервали, а також оцінити, як змінюється довжина довірчого інтервалу при збільшенні $ n $.
Інакше кажучи, виводити на друк:
- кількість виконаних реалізацій;
- отриману оцінку;
- побудований довірчий інтервал;
- ширину довірчого інтервалу.

**Зауваження**. Формули для побудови оцінок та довірчих інтервалів див. лекції 3 та 4.
Для випадку b) краще використовувати незміщену оцінку дисперсії.

## Вимоги

$ 1 - \gamma = 0.99 $  
$ n = 10^2, n = 10^4, n = 10^6 $

Для кожного $ n $ виводити на друк:
- кількість виконаних реалізацій;
- отриману оцінку;
- побудований довірчий інтервал;
- ширину довірчого інтервалу.

## Реалізація

Незміщена оцінка дисперсії: 
$$ \hat{\sigma}_n^2 = \frac{1}{n-1} \sum_{k=1}^{n}{\left(X_k - \left(\frac{1}{n}\sum_{k=1}^{n}{X_k}\right)\right)^2} \xrightarrow[n \to \infty]{} \sigma^2 $$

In [25]:
class Info:
    title = ""
    n = -1
    estimation = float("inf")
    interval = [0, 0]

    @property
    def width(self):
        return self.interval[1] - self.interval[0]

def generate_data(n):
    amount = n // 2
    w1 = np.random.uniform(0, 1, amount)
    w2 = np.random.uniform(0, 1, amount)
    
    xi_1 = np.sqrt(-2 * np.log(w1)) * np.sin(2 * np.pi * w2)
    xi_2 = np.sqrt(-2 * np.log(w1)) * np.cos(2 * np.pi * w2)
    return np.concatenate((xi_1, xi_2))

def _interval(interval, info):
    if info is not None:
        info.interval = interval
    return interval


def normal_distribution_mean_estimation_without_known_variance(n: float, gamma: float, X_mean: float, sigma: float, info: Info = None) -> list[float]:
    """
    a) Довірчий інтервал для мат. сподівання (розподіл нормальний, дисперсія невідома)

    Використовуємо квантилі розподілу Стьюдента з n-1 ступенями свободи
    """
    t_value = stats.t.ppf(1 - gamma / 2, n - 1)
    a = t_value * sigma / np.sqrt(n)
    interval = [X_mean - a, X_mean + a]
    return _interval(interval, info)

def unknown_distribution_mean_estimation(n: float, gamma: float, X_mean: float, sigma: float, info: Info = None) -> list[float]:
    """
    b) Довірчий інтервал для мат. сподівання (розподіл невідомий)

    Використовуємо асимптотичний довірчий інтервал (нормальний розподіл/функція Лапласа)
    """
    z_value = stats.norm.ppf(1 - gamma / 2)
    a = z_value * sigma / np.sqrt(n)
    interval = [X_mean - a, X_mean + a]
    return _interval(interval, info)

def normal_distribution_var_estimation(n: float, gamma: float, X_var: float, info: Info = None) -> list[float]:
    """
    c) Довірчий інтервал для дисперсії (нормальний розподіл)
    
    Використовуємо розподіл Хі-квадрат з n-1 ступенями свободи
    """
    upper = stats.chi2.ppf(1 - gamma / 2, n - 1)
    lower = stats.chi2.ppf(gamma / 2, n - 1)
    interval = [(n - 1) * X_var / upper, (n - 1) * X_var / lower]
    return _interval(interval, info)
    

def print_info(info: Info):
    width_separate = 100
    print(info.title)
    print("_"*width_separate)
    print(f"[Кількість виконаних реалізацій] n = {info.n}")
    print("_"*width_separate)
    print(f"[Отримана оцінка] {info.estimation}")
    print(f"[Довірчий інтервал] {info.interval}")
    print(f"[Ширина довірчого інтервалу] {info.width}")
    print("_"*width_separate)
    print("\n")

In [26]:
confidence_level = 0.99
gamma = 1 - confidence_level

numbers = [10**2, 10**4, 10**6]
infos = [
    # довірчий інтервал для математичного сподівання $ a $  у припущенні, що спостерігаються в.в. $ \{ X_i \} $, які мають нормальний розподіл, але дисперсія $ \sigma^2 $ невідома;
    "[Необхідно] Довірчий інтервал для математичного сподівання a при нормальному розподілі X_i але невідомій дисперсії sigma_i",
    # довірчий інтервал для математичного сподівання $ a $ у припущенні, що спостерігаються в.в. $ \{ X_i \} $, розподіл яких невідомий.
    "[Необхідно] Довірчий інтервал для математичного сподівання a при невідомому розподілі X_i",
    # довірчий інтервал для дисперсії $ \sigma^2 $ у припущенні, що спостерігаються в.в. $ \{ X_i \} $, які мають нормальний розподіл.
    "[Необхідно] Довірчий інтервал для дисперсії sigma^2 при нормальному розподілі X_i"
]

In [ ]:
info = Info()
for info_index in range(len(infos)):
    info.title = infos[info_index]
    for n in numbers:
        info.n = n
        X = generate_data(n)
        
        X_mean = np.mean(X)
        X_var = np.var(X, ddof=1)
        if info_index != 2:
            info.estimation = X_mean
        else:
            info.estimation = X_var
        sigma = np.sqrt(X_var)
        
        match info_index:
            case 0:
                normal_distribution_mean_estimation_without_known_variance(n, gamma, X_mean, sigma, info)
            case 1:
                unknown_distribution_mean_estimation(n, gamma, X_mean, sigma, info)
            case 2:
                normal_distribution_var_estimation(n, gamma, X_var, info)

        print_info(info)
    print("="*100)
    print("\n")


[Необхідно] Довірчий інтервал для математичного сподівання a при нормальному розподілі X_i але невідомій дисперсії sigma_i
____________________________________________________________________________________________________
[Кількість виконаних реалізацій] n = 100
____________________________________________________________________________________________________
[Отримана оцінка] 0.1750382094436802
[Довірчий інтервал] [np.float64(-0.08242797918143197), np.float64(0.4325043980687924)]
[Ширина довірчого інтервалу] 0.5149323772502243
____________________________________________________________________________________________________


[Необхідно] Довірчий інтервал для математичного сподівання a при нормальному розподілі X_i але невідомій дисперсії sigma_i
____________________________________________________________________________________________________
[Кількість виконаних реалізацій] n = 10000
____________________________________________________________________________________________

# Завдання 2

## Опис

Обчислення ймовірності $Q_n = P\{ \xi_1 + K + \xi_2 \}$  трьома способами із дослідженням швидкості збіжності, тобто кількості реалізацій, витрачених різними алгоритмами на побудову оцінки із заданими достовірністю та відносною похибкою (інакше кажучи, виявлення найкращого методу для тих чи інших варіантів задачі).

Випадкові величини $\{ \xi_i \}$ є незалежними та однаково розподіленими (розподіл Вейбулла):
$$ F(u) = P\{ \xi_i < u \} = 1 - e^{-u^2}, u >= 0 $$

В.в. $ \eta $  може мати один з двох розподілів:
$$ \text{A. } G(u) = P\{ \eta < u \} = 1 - \frac{1}{(1 + u)^2}, u >= 0, M \eta = \int_0^{\inf}{[1 - G(u)]du} = \int_0^{\inf}{\frac{1}{(1 + u)^2} du} = 1 $$
$$ \text{B. } G(u) = P\{ \eta < u \} = 1 - e^{-u}, u >= 0, M \eta = \int_0^{\inf}{[1 - G(u)]du} = \int_0^{\inf}{e^{-u} du} = 1 $$

Зауваження 1. Нехай $ \omega, \omega_1, \omega_2, K $ - послідовність незалежних рівномірно розподілених на відрізку $ [0, 1] $ в.в. (послідовність псевдовипадкових чисел). Тоді моделювання в.в. $ \{ \xi_i \} $ та $ \eta $ відбувається за формулами:

$ \xi_i = F^{-1}(\omega_i), \text{ тобто } 1 - e^{-\xi_i^2} = 1 - \omega_i (\omega_i \text{ та } 1 - \omega_i \text{ мають один і той же рівномірний розподіл }) \Rightarrow \xi_i = (- \ln{\omega_i})^{1/2}; $

Випадок А:
$ \eta = G^{-1}(\omega) $, тобто $ 1 - \frac{1}{(1 + \eta)^2} = 1 - \omega \Rightarrow \eta = \frac{1}{\omega^{1/2}} - 1 $;

Випадок В:
$ \eta = G^{-1}(\omega) $, тобто $ 1 - e^{-\eta} = 1 - \omega \Rightarrow \eta = -\ln{\omega} $.

Зауваження 2. Загальна схема обчислення ймовірності $ Q_n $ виглядає наступним чином. Нехай $ \hat{q}_1, \hat{q}_2, K $ - незміщені оцінки ймовірності $ Q_n $.
Позначимо
$$ $$
- незміщена оцінка для $ Q_n $ та вибіркова дисперсія, побудовані за $ N $ реалізацій того чи іншого алгоритму.

Кількість реалізацій $ N^* $, які потрібно здійснити для обчислення ймовірності $ Q_n $ із заданими достовірністю $ 1 - \gamma $ та відносною похибкою $ \epsilon $ обчислюється за формулою:
$$ N^* = \min\{ N > N_0: N >= \frac{z_{\gamma}^2 \hat{\sigma}_n^2(N)}{\epsilon^2 \hat{Q}_n^2(N)} \}, $$
де $ N_0 $ - початкова кількість реалізацій, яка потрібна для “стабілізації” вибіркової дисперсії, а $ z_{\gamma} $ - це коефіцієнт, який знаходиться з рівняння $ 2 \Phi(z) = 1 - \gamma $ ($ \Phi(z) $ - це функція Лапласа).

В усіх наведених вище випадках обчислення вести із достовірністю 0.99 та відносною похибкою 1%, тобто $ z_{\gamma} = 2.575 $ і $ \epsilon = 0.01 $. Параметр $ n $ приймає наступні значення: 1, 10, 100, 200. В.в. $ \eta $ має один з двох розподілів (випадки А та В). Для обчислення $ Q_n $ використовуємо один з трьох методів статистичного моделювання та з’ясовуємо, який з них більш доцільно використовувати.

## Методи

**Метод 1.** Стандартний метод Монте-Карло:
$ Q_n = M I(\xi_1 + K + \xi_n < \eta) $, тобто $ \hat{q}_i = I(\xi_1^{(i)} + K + \xi_n^{(i)} < \eta_i) $,
де $ I(\dot) $ - індикаторна функція (індекс $ "i" $ відноситься до $ i $-ї реалізації алгоритму).

**Метод 2.** Використовуємо співвідношення:
$$ Q_n = \int_{0}^{\inf}{[1 - G(u)] dP\{ \xi_1 + K + \xi_n < u \}} = M[1 - G(\xi_1 + K + \xi_n)], $$
тобто
   $$ \hat{q}_i = \frac{1}{\left( 1 + \xi_1^{(i)} + K + \xi_n^{(i)} \right)^2} \text{ (випадок А);}$$
   $$ \hat{q}_i = e^{- \left( \xi_1^{(i)} + K + \xi_n^{(i)} \right)} \text{ (випадок В).}$$

**Метод 3.** Цей метод ґрунтується на формулі:
$$ Q_n = \int_{0}^{\inf}{P\{ \xi_1 + K + \xi_n < u \} dG(u)} = M_{\eta} P\{ \xi_1 + K + \xi_n < \eta \}. $$
Оскільки ймовірність $ P\{ \xi_1 + K + \xi_n < u \} $ не може бути обчисленою за явною аналітичною формулою (як у попередньому випадку), то для обчислення ймовірності $ P\{ \xi_1 + K + \xi_n < u \} $ (згортка) використовуємо наступний простий алгоритм побудови оцінки в одній реалізації.
1. Моделюємо значення в.в. $ \eta $. Нехай $ \eta = u $.
2. Як початкові значення виберемо: $ T = u, \hat{q} = 1 $ (початкова оцінка ймовірності $ P\{ \xi_1 + K + \xi_n < u \} $). 
3. У циклі $ n $ разів повторюємо наступні кроки алгоритму:
   - обчислюємо $ \gamma = P\{ \xi < T \} = 1 - e^{-T^2} $;
   - як нове значення для $ \hat{q} $ обираємо $ \hat{q} \cdot \gamma $;
   - моделюємо в.в. $ \xi^* $ з розподілом
   $$ P\{ \xi^* < \nu \} = P\{ \xi < \nu \vert \xi < T \} = \frac{1 - e^{-\nu^2}}{1 - e^{-T^2}}, \nu \in [0, T], $$
   а саме $ \xi^* = \left( -\ln \left( 1 - \omega \cdot \left( 1 - e^{-T^2} \right) \right) \right)^{1/2} $, де $ \omega $ має рівномірний розподіл на $ [0, 1] $;
   - замість $ T $ обираємо $ T - \xi^* $.
4. Коли цикл завершено, як оцінку для $ Q_n $ обираємо отримане значення $ \hat{q} $.

## Вимоги

$ z_{\gamma} = 2.575 $

$ 1 - \gamma = 0.99 $

Для кожного значення параметра $ n $ (1, 10, 100, 200), для випадків А та В для кожного з трьох методів виводити на друк:
- оцінку $ \hat{Q}_n(N^*) $;
- вибіркову дисперсію $ \hat{\sigma}_n^2(N^*)$;
- відповідний довірчий інтервал;
- кількість виконаних реалізацій $ N^* $, які знадобились для побудови оцінки із достовірністю 0.99 та відносною похибкою 1%

## Реалізація